### Feature Engineering Notebook
Transform and Prepare Features for Model Training (Classification & Regression)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split
import pandas as pd

### Load Dataset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
print(f"Dataset Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
df.head()

Dataset Loaded: 891 rows × 12 columns



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Select Target Column

In [3]:
target_col = "Survived"  # NOTE: please change to your target column

y = df[target_col].copy()
n_unique = y.nunique()

if y.dtype == "object" or n_unique == 2:
    if n_unique == 2:
        task_type = "Binary Classification"
    else:
        task_type = "Multiclass Classification"
elif pd.api.types.is_integer_dtype(y) and 2 < n_unique <= 20:
    task_type = "Multiclass Classification"
else:
    task_type = "Regression"

print(f"Target Column : {target_col}")
print(f"Unique Values : {n_unique}")
print(f"Detected Task : {task_type}")
print(f"\nValue Counts:\n{y.value_counts()}")

Target Column : Survived
Unique Values : 2
Detected Task : Binary Classification

Value Counts:
Survived
0    549
1    342
Name: count, dtype: int64


### Select Feature Columns

In [4]:
exclude_cols = ["Name", "Ticket", "Cabin", "PassengerId"]  # NOTE: columns to exclude (high-cardinality / identifiers)

feature_cols = [c for c in df.columns if c != target_col and c not in exclude_cols]
X = df[feature_cols].copy()

print(f"Feature Columns ({len(feature_cols)}):")
for col in feature_cols:
    dtype = X[col].dtype
    nulls = X[col].isnull().sum()
    print(f" {col:<9} dtype={str(dtype):<8} nulls={nulls}")
print(f"\nNew DataFrame Shape: {X.shape}")

Feature Columns (7):
 Pclass    dtype=int64    nulls=0
 Sex       dtype=object   nulls=0
 Age       dtype=float64  nulls=177
 SibSp     dtype=int64    nulls=0
 Parch     dtype=int64    nulls=0
 Fare      dtype=float64  nulls=0
 Embarked  dtype=object   nulls=2

New DataFrame Shape: (891, 7)


### Encode Categorical Features

- **Label Encoding**: maps each category to an integer (good for ordinal data or tree-based models)
- **One-Hot Encoding**: creates binary columns per category (good for linear models / neural nets)

In [5]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Categorical Columns: {cat_cols}\n")

for col in cat_cols:
    print(f"  {col}: {X[col].nunique()} unique → {X[col].unique()[:10]}")

Categorical Columns: ['Sex', 'Embarked']

  Sex: 2 unique → ['male' 'female']
  Embarked: 3 unique → ['S' 'C' 'Q' nan]


3a. Label Encoding

In [6]:
label_encode_cols = ["Sex"]  # NOTE: please change to columns you want to label-encode

for col in label_encode_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"Label Encoded '{col}': {mapping}")

print(f"\nNew DataFrame Shape: {X.shape}")
X.head()

Label Encoded 'Sex': {'female': 0, 'male': 1}

New DataFrame Shape: (891, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,1,22.0,1,0,7.2500,S
1,1,0,38.0,1,0,71.2833,C
2,3,0,26.0,0,0,7.9250,S
3,1,0,35.0,1,0,53.1000,S
4,3,1,35.0,0,0,8.0500,S


3b. One-Hot Encoding

In [7]:
onehot_encode_cols = ["Embarked"]  # NOTE: please change to columns you want to one-hot encode

before_cols = X.shape[1]
X = pd.get_dummies(X, columns=onehot_encode_cols, prefix=onehot_encode_cols, drop_first=True, dtype=int)
after_cols = X.shape[1]

print(f"One-Hot Encoded: {onehot_encode_cols}")
print(f"Columns: {before_cols} → {after_cols} (+{after_cols - before_cols} new)")
print(f"\nNew DataFrame Shape: {X.shape}")
X.head()

One-Hot Encoded: ['Embarked']
Columns: 7 → 8 (+1 new)

New DataFrame Shape: (891, 8)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,3,1,22.0,1,0,7.2500,0,1
1,1,0,38.0,1,0,71.2833,0,0
2,3,0,26.0,0,0,7.9250,0,1
3,1,0,35.0,1,0,53.1000,0,1
4,3,1,35.0,0,0,8.0500,0,1


### Scale Numeric Features

- **Standard Scaler**: zero mean, unit variance (good for linear models, SVMs, neural nets)
- **MinMax Scaler**: scales to [0, 1] range

In [ ]:
scaler_type = "standard-scaler"  # NOTE: options → "standard-scaler" or "minmax-scaler"
scale_cols = X.select_dtypes(include="number").columns.tolist()  # NOTE: change to specific columns if needed

scalers = {
    "standard-scaler": StandardScaler,
    "minmax-scaler": MinMaxScaler,
}

scaler = scalers[scaler_type]()
X[scale_cols] = scaler.fit_transform(X[scale_cols])

print(f"Scaled {len(scale_cols)} Column(s) Using {scaler_type.title()} Scaler")
print(f"Columns: {scale_cols}\n")
X[scale_cols].describe().round(4)

Scaled 8 Column(s) Using Standard-Scaler Scaler
Columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']



,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
count,891.0000,891.0000,714.0000,891.0000,891.0000,891.0000,891.0000,891.0000
mean,-0.0000,-0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-0.0000
std,1.0006,1.0006,1.0007,1.0006,1.0006,1.0006,1.0006,1.0006
min,-1.5661,-1.3556,-2.0170,-0.4745,-0.4737,-0.6484,-0.3076,-1.6147
25%,-0.3694,-1.3556,-0.6595,-0.4745,-0.4737,-0.4891,-0.3076,-1.6147
50%,0.8274,0.7377,-0.1170,-0.4745,-0.4737,-0.3574,-0.3076,0.6193
75%,0.8274,0.7377,0.5718,0.4328,-0.4737,-0.0242,-0.3076,0.6193
max,0.8274,0.7377,3.4651,6.7842,6.9741,9.6672,3.2514,0.6193


### 5. Train / Test Split

In [12]:
test_size = 0.2
random_state = 42

stratify_col = y if task_type != "Regression" else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=stratify_col,
)

print(f"Train set: {X_train.shape[0]:,} samples ({1 - test_size:.0%})")
print(f"Test set:  {X_test.shape[0]:,} samples ({test_size:.0%})")
print(f"Features:  {X_train.shape[1]}")

Train set: 712 samples (80%)
Test set:  179 samples (20%)
Features:  8


### 6. Feature Engineering Summary

In [14]:
print(f"Task Type       : {task_type}")
print(f"Target Column   : {target_col}")
print(f"Feature Count   : {X.shape[1]}")
print(f"Label Encoded   : {label_encode_cols}")
print(f"One-Hot Encoded : {onehot_encode_cols}")
print(f"Scaler          : {scaler_type.title()}")
print(f"Train Samples   : {X_train.shape[0]:,}")
print(f"Test Samples    : {X_test.shape[0]:,}")
print(f"Missing Values  : {X.isnull().sum().sum()}")
print(f"\nFinal Feature Columns: {list(X.columns)}")

Task Type       : Binary Classification
Target Column   : Survived
Feature Count   : 8
Label Encoded   : ['Sex']
One-Hot Encoded : ['Embarked']
Scaler          : Standard-Scaler
Train Samples   : 712
Test Samples    : 179
Missing Values  : 177

Final Feature Columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']


### Final Engineered Dataset Preview

In [11]:
X.head(10)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0.827377,0.737695,-0.530377,0.432793,-0.473674,-0.502445,-0.307562,0.619306
1,-1.566107,-1.355574,0.571831,0.432793,-0.473674,0.786845,-0.307562,-1.614710
2,0.827377,-1.355574,-0.254825,-0.474545,-0.473674,-0.488854,-0.307562,0.619306
3,-1.566107,-1.355574,0.365167,0.432793,-0.473674,0.420730,-0.307562,0.619306
4,0.827377,0.737695,0.365167,-0.474545,-0.473674,-0.486337,-0.307562,0.619306
5,0.827377,0.737695,NaN,-0.474545,-0.473674,-0.478116,3.251373,-1.614710
6,-1.566107,0.737695,1.674039,-0.474545,-0.473674,0.395814,-0.307562,0.619306
7,0.827377,0.737695,-1.908136,2.247470,0.767630,-0.224083,-0.307562,0.619306
8,0.827377,-1.355574,-0.185937,-0.474545,2.008933,-0.424256,-0.307562,0.619306
9,-0.369365,-1.355574,-1.081480,0.432793,-0.473674,-0.042956,-0.307562,-1.614710
